# Snowpark DataFrames

Working with Snowpark dataframes is similar to Pandas Dataframes. This lets you use Python and directly connect to Snowflake tables.

The first stpe is to import the `get_active_session` function and save it to a variable. This will create a direct reference to our snowflake session. 

In [ ]:
# Get the active Snowpark session (works inside Snowflake Notebooks)
from snowflake.snowpark.functions import col, sum as sum_, avg, count
session = get_active_session()

SnowparkSessionException: (1403): No default Session is found. Please create a session before you call function 'udf' or use decorator '@udf'.

Creating a reference to the data that we want (referencing a table or a view):

In [ ]:
# Create a DataFrame reference to the menu table
df = session.table("TASTY_BYTES_SAMPLE_DATA.raw_pos.menu")

The `.show()` method lets you visualize only the data and `.collect()` when you need to work directly with the row contents of a DataFrame (extract te and modify them)

In [ ]:
# Display the first 10 rows (default)
df.show()

In [ ]:
from snowflake.snowpark.functions import col

# Create a DataFrame reference to the menu table
df = session.table("TASTY_BYTES_SAMPLE_DATA.RAW_POS.MENU")
df_macshack = df.filter(col("TRUCK_BRAND_NAME") == "The Mac Shack").select(col("MENU_ITEM_NAME"), col("ITEM_CATEGORY"))
df_macshack.show()


In [ ]:
# .collect() pulls ALL rows into local memory as a list of Row objects
rows = df.collect()
print(f"Row count: {len(rows)}")
print(rows[0])

The following codes will make use of different functions to select and filter a table and ALL of them will output the same result. The only difference is the code:

In [ ]:
from snowflake.snowpark.functions import col

# Filter the DataFrame to only include rows where truck brand is "Freezing Point"
df_freezing = df.filter(col("TRUCK_BRAND_NAME") == "Freezing Point")
df_freezing.show()

# Select only specific columns from the filtered DataFrame
df_selected = df_freezing.select(
    col("MENU_ITEM_NAME"),
    col("SALE_PRICE_USD")
)
df_selected.show()

In [ ]:
# Method chaining: filter and select in one fluent expression
df_result = (
    df
    .filter(col("TRUCK_BRAND_NAME") == "Freezing Point")
    .select(col("MENU_ITEM_NAME"), col("SALE_PRICE_USD"))
)
df_result.show()

In [ ]:
# Alternative: use raw SQL via session.sql() for the same result
df_sql = session.sql("""
    SELECT MENU_ITEM_NAME, SALE_PRICE_USD
    FROM TASTY_BYTES_SAMPLE_DATA.raw_pos.menu
    WHERE TRUCK_BRAND_NAME = 'Freezing Point'
""")
df_sql.show()

We can also create a Snowpark Dataframe directly in Python:

In [ ]:
# Create a DataFrame from local Python data (useful for small test datasets)
data = [("lemonade", 4.50), ("ice cream sandwich", 6.00), ("bottled water", 2.00)]
df_local = session.create_dataframe(data, schema=["item", "price"])
df_local.show()

In the next cell, we will be using snowflake's built in functions to calculate aggregation metrics and group by. *Note that the "sum" operator is named* `sum_` *so that it does not overshadows the* `sum` *method built in Python.*

In [ ]:
# Aggregate: group by truck brand and compute count, average price, and total menu value
df_summary = (
    df
    .group_by(col("TRUCK_BRAND_NAME"))
    .agg(
        count(col("MENU_ITEM_NAME")).alias("item_count"),
        avg(col("SALE_PRICE_USD")).alias("avg_price"),
        sum_(col("SALE_PRICE_USD")).alias("total_menu_value")
    )
    .sort(col("avg_price").desc())
)
df_summary.show()

There is also a different way to do that, and is by using the "describe()" method.

In [ ]:
# .describe() computes summary stats (count, mean, stddev, min, max) for selected columns
df.select(col("SALE_PRICE_USD"), col("COST_OF_GOODS_USD")).describe().show()

Here, we can convert an Snowpark DF to an Pandas DF:

In [ ]:
# Convert Snowpark DataFrame to a Pandas DataFrame (pulls data to local memory)
df_pandas = df_summary.to_pandas()
print(type(df_pandas))
df_pandas.head()

Here is the same conversion but in reverse

In [ ]:
# Create the analytics schema if it doesn't already exist
session.sql("CREATE SCHEMA IF NOT EXISTS TASTY_BYTES_SAMPLE_DATA.analytics").collect()

# Write the summary DataFrame to a Snowflake table (overwrite if it already exists)
df_summary.write.mode("overwrite").save_as_table(
    "TASTY_BYTES_SAMPLE_DATA.analytics.brand_price_summary"
)

# Verify the table was created by reading it back
session.table("TASTY_BYTES_SAMPLE_DATA.analytics.brand_price_summary").show()

# Using Snowpark Dataframes outside of Snowflake Notebooks (Like in VS Code)

It is possible to do this, but in order to do so, one has to create a session manually with Snowflake credentials. Also, the library `snowflake-snowpark-python` library needs to be installed and it is a good practice to save the credentials as an environment variable and not in a text file.

Shell Install Command For Snowpark Python Library

`pip install snowflake-snowpark-python`

Tip: This install is only needed when running Snowpark outside of Snowflake (e.g., in a local IDE). In Snowflake Notebooks, Snowpark is pre-installed and ready to use.

In [1]:
import os
from dotenv import load_dotenv
from snowflake.snowpark import Session
from snowflake.snowpark.functions import col, sum as sum_, avg, count

I will use a **environment variables**, in order to safely sign in to Snowflake and to load my Claude API. *[...]*

In [2]:
# Change folder to current directory of the Jupyter Notebook
current_path=os.getcwd()
os.chdir(current_path)

# Load environment variables from .env file
load_dotenv(override=True) # override=True para que no haya conflictos con la Claude API Key en el .env 

True

In [3]:
# Connection parameters for establishing a Snowpark session
# These environment variables must be set with real credentials before running
connection_params = {
    "account": os.getenv("SNOWFLAKE_ACCOUNT"),
    "user": os.getenv("SNOWFLAKE_USER"),
    "password": os.getenv("SNOWFLAKE_PASSWORD"),
    "role": os.getenv("SNOWFLAKE_ROLE"),
    "warehouse": os.getenv("SNOWFLAKE_WAREHOUSE"),
    "database": os.getenv("SNOWFLAKE_DATABASE"),
    "schema": os.getenv("SNOWFLAKE_SCHEMA"), 
    # "passcode": 901937 # MFA passcode, this is optional if you have an autheticator for your Snowflake Account
}

# Create a Snowpark session using the connection parameters
session = Session.builder.configs(connection_params).create()

# Create a DataFrame reference to the menu table (lazy evaluation – no data fetched yet)
df = session.table("TASTY_BYTES.raw_pos.menu")

# Apply a filter and display results; .show() triggers execution and prints output
df.filter(col("TRUCK_BRAND_NAME") == "Freezing Point").show()

---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"MENU_ID"  |"MENU_TYPE_ID"  |"MENU_TYPE"  |"TRUCK_BRAND_NAME"  |"MENU_ITEM_ID"  |"MENU_ITEM_NAME"    |"ITEM_CATEGORY"  |"ITEM_SUBCATEGORY"  |"COST_OF_GOODS_USD"  |"SALE_PRICE_USD"  |"MENU_ITEM_HEALTH_METRICS_OBJ"     |
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|10001      |1               |Ice Cream    |Freezing Point      |10              |Lemonade            |Beverage         |Cold Option         |0.6500               |3.5000            |{                                  |
|           |                |             |                    |                |                    |                 

## Testing Claude API connection

In [3]:
import anthropic

client = anthropic.Anthropic()

In [ ]:
message = client.messages.create(
    model="claude-haiku-5-5",
    max_tokens=1000,
    messages=[
        {
            'role': 'user',
            'content': 'Describe briefly the Tasty Bytes Dataset provided by Snowflake? Please respond using a maximum of 10 lines.',
        }
    ]
)

for block in message.content:
    if block.type == "text":
        print(block.text)

The **Tasty Bytes** dataset is a fictional, globally distributed food truck company that Snowflake provides for demos, tutorials, and quickstarts. It includes sample data on menus, trucks, locations, point-of-sale orders, and customer loyalty, spanning many cities across multiple countries. Because it's realistic but synthetic, it lets users practice data loading, transformation, analytics, and AI/ML workflows in Snowflake without using real business data.


## Making some tests with Claude API

In [ ]:
# functions for multi-turn conversations

def add_user_msg(msgs, txt):
    # Adds an user message to the history
    user_msg = {'role': 'user', 'content':txt}
    msgs.append(user_msg)

def add_assistant_msg(msgs, txt):
    # Adds an assistant (AI) message to the history
    assistant_msg = {'role': 'assistant', 'content':txt}
    msgs.append(assistant_msg)

def chat(msgs, model, max_tokens=1000, system_prompt=None, output_format=None):
    # Creates a chat with the corresponding message history
    params = {
        'model': model,
        'max_tokens': max_tokens,
        'messages': msgs
    }
    if system_prompt:
        params['system'] = system_prompt
    if output_format:
        params['output_config'] = {'format': output_format}

    output = client.messages.create(**params)
    # Joining all text blocks, and checking if the block is a TextBlock
    return "".join(b.text for b in output.content if b.type == "text")

In [6]:
# test (BORRAR DESPUÉS)
messages = []

add_user_msg(messages, 'Define what a relational database is in one sentence')

ai_answer = chat(messages, model='claude-sonnet-5-5')

add_assistant_msg(messages, ai_answer)

add_user_msg(messages, 'Write another sentence')

final_answer = chat(messages, model='claude-sonnet-5-5')
final_answer

'A relational database stores information in structured tables that link to one another through primary and foreign keys, which lets you retrieve and combine related data efficiently while keeping it consistent and free of unnecessary duplication.'

## Starting to write the Claude-Snowflake API Pipeline

Re-writing relevant code for the pipliine.

In [ ]:
import os
from dotenv import load_dotenv
from snowflake.snowpark import Session
from snowflake.snowpark.functions import col, sum as sum_, avg, count

import json
from pathlib import Path

import anthropic

# Change folder to current directory of the Jupyter Notebook
current_path=os.getcwd()
os.chdir(current_path)

# Load environment variables from .env file
load_dotenv(override=True) # override=True para que no haya conflictos con la Claude API Key en el .env 

# COnnecting to Claude API
client = anthropic.Anthropic()

In [20]:
def add_user_msg(msgs, txt):
    # Adds an user message to the history
    user_msg = {'role': 'user', 'content':txt}
    msgs.append(user_msg)

def add_assistant_msg(msgs, txt):
    # Adds an assistant (AI) message to the history
    assistant_msg = {'role': 'assistant', 'content':txt}
    msgs.append(assistant_msg)

def chat_and_tokens(msgs, model, max_tokens=1000, system_prompt=None, output_format=None):
    # Creates a chat with the corresponding message history, and outputs the input and output tokens used
    params = {
        'model': model,
        'max_tokens': max_tokens,
        'messages': msgs
    }
    if system_prompt:
        params['system'] = system_prompt
    if output_format:
        params['output_config'] = {'format': output_format}

    output = client.messages.create(**params)
    # Joining all text blocks, and checking if the block is a TextBlock
    return ("".join(b.text for b in output.content if b.type == "text"), output.usage.input_tokens, output.usage.output_tokens)

In [21]:
sql_schema = {
    "type": "json_schema",
    "schema": {
        "anyOf": [
            { # Output Schema in case the user question is simple or doesnt need a clarification
                "type": "object",
                "properties": {
                    "question_type": {"type": "string", "const": "sql"},
                    "query": {
                        "type": "string",
                        "description": "Query SQL ejecutable que responde la pregunta.",
                    },
                    "explanation": {
                        "type": "string",
                        "description": "Explicación breve de la query, máximo 3 oraciones.",
                    },
                },
                "required": ["question_type", "query", "explanation"],
                "additionalProperties": False,
            },
            { # Output schema in case the user question needs a clarification
                "type": "object",
                "properties": {
                    "question_type": {"type": "string", "const": "clarify"},
                    "message": {
                        "type": "string",
                        "description": "Pregunta breve de aclaración para el usuario.",
                    },
                },
                "required": ["question_type", "message"],
                "additionalProperties": False,
            },
        ]
    },
}

# Opening the JSON file with the Evaluation Tests
with open('eval_dataset.json', 'r') as file:
    test_cases = json.load(file)

test = test_cases[0]['task'] #Using just the first test case for now

user_question = f"Write an SQL query in order to answer the user's question about Tasty Bytes: {test}" # User question un natural language
# Later, this variable will recieve the input of the streamlit app, like:
# user_question = input('Ask a question about Tasty Bytes.') 

messages = []
add_user_msg(messages, user_question) # Adding the user question prompt to the message history

system = Path('claude_prompt_original.md').read_text(encoding='utf-8') #Loading the MD file with the system prompt with all the specifications

ai_answer, input_tokens, output_tokens = chat_and_tokens( # Calling Claude API to write the SQL query
    messages,
    model='claude-sonnet-5-5',
    max_tokens=1000,
    system_prompt=system,
    output_format=sql_schema,
)

add_assistant_msg(messages, ai_answer) # Saving the AI response in the message history [CHECAR DESPUÉS SI LE METO TODO AL CHAT EN FORMATO JSON O EN OTRO FORMATO] 

result = json.loads(ai_answer) # COnverting the string generated by Claude into an JSON
print(result["query"]) # Printing the query
print('\n# of Input Tokens used:',input_tokens) # How much input tokens were used?
print('# of Ouput Tokens used:',output_tokens) # How much output tokens were used?

WITH order_totals AS (
    -- Total per order: sum the line items to avoid per-item averages
    SELECT
        o.truck_brand_name,
        o.order_id,
        SUM(o.price) AS order_total,
    FROM analytics.orders_v AS o
    WHERE o.date >= '2021-01-01'
        AND o.date <= '2021-12-31'
        AND o.country = 'Canada'
    GROUP BY
        o.truck_brand_name,
        o.order_id
)
SELECT
    ot.truck_brand_name,
    AVG(ot.order_total) AS avg_order_value,
FROM order_totals AS ot
GROUP BY ot.truck_brand_name
ORDER BY avg_order_value DESC;

# of Input Tokens used: 3881
# of Ouput Tokens used: 978


In [22]:
print(result['explanation'])

Primero se suma el precio de los items de cada pedido en `analytics.orders_v` para obtener el total por pedido, filtrando Canadá y el año 2021. Luego se promedian esos totales por marca de camión (`truck_brand_name`), evitando promediar por item. Los resultados se ordenan de mayor a menor valor promedio de pedido.


In [23]:
# Connection parameters for establishing a Snowpark session
# These environment variables must be set with real credentials before running
connection_params = {
    "account": os.getenv("SNOWFLAKE_ACCOUNT"),
    "user": os.getenv("SNOWFLAKE_USER"),
    "password": os.getenv("SNOWFLAKE_PASSWORD"),
    "role": os.getenv("SNOWFLAKE_ROLE"),
    "warehouse": os.getenv("SNOWFLAKE_WAREHOUSE"),
    "database": os.getenv("SNOWFLAKE_DATABASE"),
    "schema": os.getenv("SNOWFLAKE_SCHEMA")
}

# Create a Snowpark session using the connection parameters
session = Session.builder.configs(connection_params).create()

In [ ]:
# Running the SQL query 
df = session.sql(result["query"]).collect()
df

[Row(TRUCK_BRAND_NAME='Kitakata Ramen Bar', AVG_ORDER_VALUE=Decimal('60.3789671877')),
 Row(TRUCK_BRAND_NAME="Nani's Kitchen", AVG_ORDER_VALUE=Decimal('58.5519543767')),
 Row(TRUCK_BRAND_NAME='Cheeky Greek', AVG_ORDER_VALUE=Decimal('50.6074296993')),
 Row(TRUCK_BRAND_NAME='Better Off Bread', AVG_ORDER_VALUE=Decimal('48.2737312560')),
 Row(TRUCK_BRAND_NAME='Smoky BBQ', AVG_ORDER_VALUE=Decimal('45.0672895264')),
 Row(TRUCK_BRAND_NAME='Revenge of the Curds', AVG_ORDER_VALUE=Decimal('44.9415332812')),
 Row(TRUCK_BRAND_NAME='Le Coin des Crêpes', AVG_ORDER_VALUE=Decimal('42.7187595785')),
 Row(TRUCK_BRAND_NAME='The Mac Shack', AVG_ORDER_VALUE=Decimal('39.3214298760')),
 Row(TRUCK_BRAND_NAME='Tasty Tibs', AVG_ORDER_VALUE=Decimal('39.2718420128')),
 Row(TRUCK_BRAND_NAME='Peking Truck', AVG_ORDER_VALUE=Decimal('35.9111260366')),
 Row(TRUCK_BRAND_NAME='Plant Palace', AVG_ORDER_VALUE=Decimal('33.7247120058')),
 Row(TRUCK_BRAND_NAME='Amped Up Franks', AVG_ORDER_VALUE=Decimal('32.5017838186')),
 Ro

## Prompt Engineering

## Prompt Evaluation